# Part II · Limitation L2 — Non-comparable Per-Block Thresholds
**Task 9** | *Worksheet §Part II, Limitation L2 (30 min)*

## The problem

When no global threshold is given, each block $b$ computes its own
$$\theta_b = Q_q(\mathcal{D}_b^+)$$
from the positive signal increments **within that block alone**.

If a mutant cell line has systematically larger ERK pulses, its $\theta_b$ is also larger —
so we end up comparing "top-10% steps in WT" to "top-10% steps in PTEN-del",
which are intrinsically different events and mask the biological amplitude difference.

## The fix: global threshold $\theta^*$

Pool **all** positive increments across **all** blocks:
$$\theta^* = Q_q\!\left(\bigcup_{b} \mathcal{D}_b^+\right)$$
Re-run every block with this fixed $\theta^*$ (via `--jump-threshold`).
Now the same increment size counts as a jump everywhere.

In [ ]:
import sys, json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
SCRIPTS_DIR  = PROJECT_ROOT / 'scripts'
DATA_PATH    = PROJECT_ROOT / 'single-cell-tracks_exp1-6_noErbB2.csv.gz'
META_PATH    = PROJECT_ROOT / '01-readme-experiment-description_2022-04-05.csv'
OUTPUT_ROOT  = PROJECT_ROOT / 'analysis_outputs'

SIGNAL_COL = 'ERKKTR_ratio'
QUANTILE   = 0.90

# Sites we will compare (one per mutation to keep runtimes manageable)
SITES = [
    {'exp_id': 1, 'site_id': 1,  'mutation': 'WT'},
    {'exp_id': 1, 'site_id': 5,  'mutation': 'AKT1_E17K'},
    {'exp_id': 1, 'site_id': 9,  'mutation': 'PIK3CA_E545K'},
    {'exp_id': 1, 'site_id': 13, 'mutation': 'PIK3CA_H1047R'},
    {'exp_id': 1, 'site_id': 17, 'mutation': 'PTEN_del'},
]

print('Sites to analyse:', [s['mutation'] for s in SITES])

---
## Step 0 — Run each block with per-block threshold (baseline)

This reproduces the default behaviour so we have something to compare against.
Blocks already computed in Part I are reused from disk.

In [ ]:
def run_block(exp, site, signal, outdir, threshold=None, quantile=0.9):
    cmd = [
        sys.executable, str(SCRIPTS_DIR / 'spatiotemporal_signal_propagation.py'),
        '--data-path', str(DATA_PATH), '--meta-path', str(META_PATH),
        '--exp-id', str(exp), '--site-id', str(site),
        '--signal-col', signal,
        '--spatial-radius', '60',
        '--future-window-frames', '3',
        '--jump-quantile', str(quantile),
        '--output-dir', str(outdir),
    ]
    if threshold is not None:
        cmd += ['--jump-threshold', str(threshold)]
    subprocess.run(cmd, capture_output=True)
    sp = outdir / f'exp_{exp}_site_{site}_{signal}' / 'summary.json'
    with open(sp) as f:
        return json.load(f)


per_block_results = []
for site_info in SITES:
    exp, site, mut = site_info['exp_id'], site_info['site_id'], site_info['mutation']
    sp = OUTPUT_ROOT / f'exp_{exp}_site_{site}_{SIGNAL_COL}' / 'summary.json'
    if sp.exists():
        print(f'  {mut}: reusing existing output')
        with open(sp) as f:
            s = json.load(f)
    else:
        print(f'  {mut}: running …', end=' ', flush=True)
        s = run_block(exp, site, SIGNAL_COL, OUTPUT_ROOT)
        print('done')
    s['mutation'] = mut
    per_block_results.append(s)

df_per = pd.DataFrame(per_block_results)[['mutation','jump_threshold',
    'future_jump_rate_if_neighbor_jumps_now',
    'future_jump_rate_if_no_neighbor_jumps_now','relative_risk']].rename(columns={
    'jump_threshold': 'θ_per_block',
    'future_jump_rate_if_neighbor_jumps_now': 'p_exp',
    'future_jump_rate_if_no_neighbor_jumps_now': 'p_unexp',
    'relative_risk': 'RR_per_block',
})
df_per

---
## Step 1 — Visualise the distribution of per-block thresholds

If thresholds vary substantially between mutations, cross-group $\mathrm{RR}$ is not comparable.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
colors  = plt.cm.tab10.colors
ax.bar(df_per['mutation'], df_per['θ_per_block'], color=colors[:len(df_per)])
ax.set_ylabel('Per-block threshold $\\theta_b$')
ax.set_title('Per-block jump thresholds — are they comparable?')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Which mutation has the highest θ_b? What does this imply about its ERK pulse amplitude?


---
## Step 2 — Pool positive increments and compute $\theta^*$

In [ ]:
all_deltas = []

for site_info in SITES:
    exp, site = site_info['exp_id'], site_info['site_id']
    npath = OUTPUT_ROOT / f'exp_{exp}_site_{site}_{SIGNAL_COL}' / 'nodes.csv.gz'
    if npath.exists():
        df_n = pd.read_csv(npath, usecols=['signal_delta'])
        pos  = df_n['signal_delta'].dropna()
        pos  = pos[pos > 0].values
        all_deltas.append(pos)
        print(f"  site {site}: {len(pos):,} positive increments, median {np.median(pos):.4f}")
    else:
        print(f"  site {site}: nodes.csv.gz not found — run Block 3 first")

all_deltas_pooled = np.concatenate(all_deltas)
theta_star = float(np.quantile(all_deltas_pooled, QUANTILE))
print(f"\nGlobal threshold θ* = {theta_star:.4f}  (q={QUANTILE})")

In [ ]:
# Plot the pooled distribution with θ* marked
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(all_deltas_pooled, bins=80, color='steelblue', alpha=0.7,
        range=(0, np.quantile(all_deltas_pooled, 0.99)))
ax.axvline(theta_star, color='red', lw=2, ls='--', label=f'θ* = {theta_star:.3f}')
ax.set_xlabel('Positive signal increment $\\Delta S_k(t)$')
ax.set_ylabel('Count')
ax.set_title('Pooled positive increments — global threshold')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 3 — Re-run all blocks with global threshold $\theta^*$

In [ ]:
GLOBAL_OUTDIR = OUTPUT_ROOT / 'global_threshold'

global_results = []
for site_info in SITES:
    exp, site, mut = site_info['exp_id'], site_info['site_id'], site_info['mutation']
    print(f'  {mut} …', end=' ', flush=True)
    s = run_block(exp, site, SIGNAL_COL, GLOBAL_OUTDIR, threshold=theta_star)
    s['mutation'] = mut
    global_results.append(s)
    print('done')

df_global = pd.DataFrame(global_results)[['mutation','jump_threshold',
    'future_jump_rate_if_neighbor_jumps_now',
    'future_jump_rate_if_no_neighbor_jumps_now','relative_risk']].rename(columns={
    'jump_threshold': 'θ_global',
    'future_jump_rate_if_neighbor_jumps_now': 'p_exp',
    'future_jump_rate_if_no_neighbor_jumps_now': 'p_unexp',
    'relative_risk': 'RR_global',
})
df_global

---
## Step 4 — Compare per-block vs. global threshold

In [ ]:
comparison = df_per[['mutation','θ_per_block','RR_per_block']].merge(
    df_global[['mutation','RR_global']], on='mutation'
)
comparison['ΔRR'] = comparison['RR_global'] - comparison['RR_per_block']
print(comparison.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(comparison))
ax.bar([i - 0.2 for i in x], comparison['RR_per_block'], width=0.4,
       label='Per-block θ', color='steelblue', alpha=0.85)
ax.bar([i + 0.2 for i in x], comparison['RR_global'],    width=0.4,
       label=f'Global θ* = {theta_star:.3f}', color='darkorange', alpha=0.85)
ax.axhline(1, color='grey', ls='--', lw=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(comparison['mutation'], rotation=20, ha='right')
ax.set_ylabel('Relative risk RR')
ax.set_title('Per-block vs. global threshold: impact on RR')
ax.legend()
plt.tight_layout()
plt.show()

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# (a) Does the mutation ranking change between per-block and global thresholds?
# (b) For which mutation is the bias largest (|ΔRR| biggest)?
# (c) Is the per-block estimator systematically inflated or deflated?


In [ ]:
# ── YOUR CODE ────────────────────────────────────────────────────────────────
# Compare the fraction of nodes labelled as jumps under each threshold.
# Load nodes from OUTPUT_ROOT and GLOBAL_OUTDIR, compute jump_event.mean() per site.
# Which mutation has the largest change in jump frequency?
